# Stress Testing y Simulación de Escenarios

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Simular escenarios macroeconómicos** con funciones SQL avanzadas
2. **Calcular impacto en cartera crediticia** bajo diferentes escenarios de estrés
3. **Usar `SNOWFLAKE.ML.FORECAST`** para proyectar tasas de morosidad
4. **Generar informes narrativos** con `CORTEX.COMPLETE()`
5. **Construir dashboard de escenarios** con comparativas

---

## Lo Que Construirás

Un **sistema de stress testing** que simula el impacto de escenarios adversos:
- 3 escenarios: Base, Adverso, Severamente Adverso
- Proyección de morosidad con ML.FORECAST
- Cálculo de pérdidas esperadas y provisiones
- Narrativas automatizadas de resultados
- Dashboard comparativo de escenarios

**Valor de Negocio**: Cumplir con requisitos regulatorios (EBA, BCE) de stress testing y anticipar necesidades de capital.

---

## Requisitos Técnicos

- **Cuenta Snowflake** con Cortex ML y Cortex AI habilitados
- **Aproximadamente 15 minutos** para completar
- **100% SQL** (excepto dashboard Streamlit)

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.FORECAST` — Proyección de series temporales
- `CORTEX.COMPLETE()` — Generación de informes narrativos
- `GENERATOR()` — Simulación de escenarios macroeconómicos
- Window functions — Análisis de tendencias y volatilidad
- CTEs encadenados — Cálculos complejos de provisiones

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Stress Testing Regulatorio

**Objetivo**: Evaluar la resiliencia de la cartera crediticia ante escenarios macroeconómicos adversos.

### Escenarios EBA

| Escenario | PIB | Desempleo | Tipo Interés |
|---|---|---|---|
| Base | +2.1% | 11.5% | 3.5% |
| Adverso | -1.5% | 15.0% | 5.0% |
| Severamente Adverso | -4.0% | 20.0% | 7.0% |

In [ ]:
CREATE DATABASE IF NOT EXISTS STRESS_TESTING_DB;
CREATE SCHEMA IF NOT EXISTS STRESS_TESTING_DB.PUBLIC;
USE SCHEMA STRESS_TESTING_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema;

---

## Paso 2: Definir Estructura de Datos

### Tablas

1. **`CARTERA_CREDITOS`** — Préstamos vivos con características
2. **`SERIES_MACRO`** — Indicadores macroeconómicos históricos mensuales
3. **`MOROSIDAD_HISTORICA`** — Tasas de morosidad por segmento y mes

In [ ]:
CREATE OR REPLACE TABLE CARTERA_CREDITOS (
    credito_id VARCHAR(10) PRIMARY KEY,
    segmento VARCHAR(30),
    importe_pendiente DECIMAL(12,2),
    tipo_interes DECIMAL(5,3),
    fecha_vencimiento DATE,
    ltv_ratio FLOAT,
    dti_ratio FLOAT,
    calificacion VARCHAR(5)
);

CREATE OR REPLACE TABLE SERIES_MACRO (
    fecha DATE PRIMARY KEY,
    pib_var_interanual FLOAT,
    tasa_desempleo FLOAT,
    euribor_12m FLOAT,
    ipc_interanual FLOAT,
    indice_vivienda FLOAT
);

CREATE OR REPLACE TABLE MOROSIDAD_HISTORICA (
    fecha DATE,
    segmento VARCHAR(30),
    tasa_morosidad FLOAT,
    provisiones DECIMAL(15,2),
    PRIMARY KEY (fecha, segmento)
);

SELECT 'Tablas creadas' AS status;

---

## Paso 3: Generar Datos Históricos

### Qué Vamos a Crear

- **10.000 créditos** en cartera (Hipotecas, Consumo, Empresas, Tarjetas)
- **36 meses** de datos macroeconómicos históricos
- **36 meses** de tasas de morosidad por segmento

In [ ]:
-- Generar cartera de créditos
INSERT INTO CARTERA_CREDITOS
SELECT
    'PRE' || LPAD(SEQ4()::STRING, 6, '0'),
    ARRAY_CONSTRUCT('Hipotecas','Consumo','Empresas','Tarjetas')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    ROUND(UNIFORM(5000, 500000, RANDOM()), 2),
    ROUND(UNIFORM(1.5, 8.0, RANDOM()), 3),
    DATEADD('month', UNIFORM(6, 360, RANDOM()), CURRENT_DATE()),
    ROUND(UNIFORM(0.20, 0.95, RANDOM()), 2),
    ROUND(UNIFORM(0.10, 0.60, RANDOM()), 2),
    ARRAY_CONSTRUCT('AAA','AA','A','BBB','BB','B','CCC')[UNIFORM(0,6,RANDOM())]::VARCHAR
FROM TABLE(GENERATOR(ROWCOUNT => 10000));

-- Generar series macroeconómicas (36 meses)
INSERT INTO SERIES_MACRO
SELECT
    DATEADD('month', -SEQ4(), CURRENT_DATE())::DATE,
    ROUND(2.0 + (RANDOM() % 200) / 100.0 - 1.0, 2),
    ROUND(11.0 + (RANDOM() % 300) / 100.0, 2),
    ROUND(3.0 + (RANDOM() % 200) / 100.0, 3),
    ROUND(2.5 + (RANDOM() % 200) / 100.0, 2),
    ROUND(100 + (RANDOM() % 2000) / 100.0, 1)
FROM TABLE(GENERATOR(ROWCOUNT => 36));

-- Generar morosidad histórica por segmento
INSERT INTO MOROSIDAD_HISTORICA
WITH meses AS (SELECT DATEADD('month', -SEQ4(), CURRENT_DATE())::DATE AS fecha FROM TABLE(GENERATOR(ROWCOUNT => 36))),
segmentos AS (SELECT column1 AS segmento FROM VALUES ('Hipotecas'),('Consumo'),('Empresas'),('Tarjetas'))
SELECT
    m.fecha, s.segmento,
    CASE s.segmento
        WHEN 'Hipotecas' THEN ROUND(2.5 + UNIFORM(-50,50,RANDOM())/100.0, 2)
        WHEN 'Consumo' THEN ROUND(5.0 + UNIFORM(-100,100,RANDOM())/100.0, 2)
        WHEN 'Empresas' THEN ROUND(4.0 + UNIFORM(-80,80,RANDOM())/100.0, 2)
        WHEN 'Tarjetas' THEN ROUND(6.0 + UNIFORM(-120,120,RANDOM())/100.0, 2)
    END,
    ROUND(UNIFORM(500000, 5000000, RANDOM()), 2)
FROM meses m CROSS JOIN segmentos s;

SELECT 'Datos generados' AS status, 
    (SELECT COUNT(*) FROM CARTERA_CREDITOS) AS creditos,
    (SELECT COUNT(*) FROM SERIES_MACRO) AS meses_macro,
    (SELECT COUNT(*) FROM MOROSIDAD_HISTORICA) AS registros_morosidad;

---

## Paso 4: Proyectar Morosidad con ML.FORECAST

### Qué Vamos a Hacer

Usar `SNOWFLAKE.ML.FORECAST` para proyectar tasas de morosidad a 12 meses.

### Entendiendo ML.FORECAST

```sql
CREATE SNOWFLAKE.ML.FORECAST model_name(
    INPUT_DATA => ...,
    TIMESTAMP_COLNAME => 'fecha',
    TARGET_COLNAME => 'tasa_morosidad',
    SERIES_COLNAME => 'segmento'  -- Forecast por cada segmento
);
```

### ¿Por Qué ML.FORECAST?

- **Multivariate**: Puede usar variables exógenas (macro)
- **Multi-series**: Proyecta cada segmento independientemente
- **AutoML**: Selecciona automáticamente el mejor algoritmo temporal

In [ ]:
-- Entrenar modelo de forecast de morosidad
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST forecast_morosidad(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'MOROSIDAD_HISTORICA'),
    TIMESTAMP_COLNAME => 'FECHA',
    TARGET_COLNAME => 'TASA_MOROSIDAD',
    SERIES_COLNAME => 'SEGMENTO'
);

-- Proyectar 12 meses
CALL forecast_morosidad!FORECAST(FORECASTING_PERIODS => 12);

---

## Paso 5: Simular Escenarios de Estrés

### Tres Escenarios

- **Base**: Continuación de tendencias actuales
- **Adverso**: Recesión moderada (PIB -1.5%, desempleo 15%)
- **Severamente Adverso**: Crisis severa (PIB -4%, desempleo 20%)

### Multiplicadores de Impacto

La tasa de morosidad se ajusta según el escenario:
- Base: factor 1.0x (sin cambio)
- Adverso: factor 1.5x — 2.0x según segmento
- Severo: factor 2.5x — 3.5x según segmento

In [ ]:
-- Simular escenarios de estrés
CREATE OR REPLACE TABLE ESCENARIOS_STRESS AS
WITH base_forecast AS (
    SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
),
segmentos AS (
    SELECT DISTINCT segmento, AVG(tasa_morosidad) AS mora_base 
    FROM MOROSIDAD_HISTORICA GROUP BY 1
)
SELECT
    s.segmento,
    'Base' AS escenario,
    s.mora_base AS tasa_mora_proyectada,
    SUM(c.importe_pendiente) * s.mora_base / 100 AS perdida_esperada,
    SUM(c.importe_pendiente) AS exposicion_total
FROM segmentos s
JOIN CARTERA_CREDITOS c ON c.segmento = s.segmento
GROUP BY s.segmento, s.mora_base

UNION ALL

SELECT
    s.segmento,
    'Adverso',
    ROUND(s.mora_base * CASE s.segmento
        WHEN 'Hipotecas' THEN 1.8 WHEN 'Consumo' THEN 2.0
        WHEN 'Empresas' THEN 1.5 ELSE 2.2 END, 2),
    SUM(c.importe_pendiente) * s.mora_base * CASE s.segmento
        WHEN 'Hipotecas' THEN 1.8 WHEN 'Consumo' THEN 2.0
        WHEN 'Empresas' THEN 1.5 ELSE 2.2 END / 100,
    SUM(c.importe_pendiente)
FROM segmentos s
JOIN CARTERA_CREDITOS c ON c.segmento = s.segmento
GROUP BY s.segmento, s.mora_base

UNION ALL

SELECT
    s.segmento,
    'Severamente Adverso',
    ROUND(s.mora_base * CASE s.segmento
        WHEN 'Hipotecas' THEN 3.0 WHEN 'Consumo' THEN 3.5
        WHEN 'Empresas' THEN 2.5 ELSE 3.8 END, 2),
    SUM(c.importe_pendiente) * s.mora_base * CASE s.segmento
        WHEN 'Hipotecas' THEN 3.0 WHEN 'Consumo' THEN 3.5
        WHEN 'Empresas' THEN 2.5 ELSE 3.8 END / 100,
    SUM(c.importe_pendiente)
FROM segmentos s
JOIN CARTERA_CREDITOS c ON c.segmento = s.segmento
GROUP BY s.segmento, s.mora_base;

SELECT escenario, segmento, tasa_mora_proyectada, 
    ROUND(perdida_esperada, 0) AS perdida_esperada,
    ROUND(exposicion_total, 0) AS exposicion
FROM ESCENARIOS_STRESS ORDER BY escenario, segmento;

---

## Paso 6: Informe Narrativo con IA

### Generación Automática de Informes

Usamos `CORTEX.COMPLETE()` para generar un resumen ejecutivo de los resultados del stress test.

In [ ]:
-- Generar informe narrativo con IA
SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
    'Eres un analista de riesgo bancario senior. Genera un resumen ejecutivo de stress testing con estos resultados:\n\n' ||
    (SELECT LISTAGG(
        'Escenario: ' || escenario || ' | Segmento: ' || segmento || 
        ' | Mora: ' || tasa_mora_proyectada || '% | Pérdida: ' || ROUND(perdida_esperada,0) || '€',
        '\n'
    ) FROM ESCENARIOS_STRESS) ||
    '\n\nGenera un resumen de 5 párrafos en español con conclusiones y recomendaciones de capital.'
) AS informe_stress_test;

---

## Paso 7: Dashboard Comparativo

### Panel de Escenarios

Dashboard interactivo para comparar impacto entre escenarios y segmentos.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Stress Testing - Simulación de Escenarios')
st.markdown('### Análisis de Impacto en Cartera Crediticia')

df = session.sql('SELECT * FROM ESCENARIOS_STRESS').to_pandas()

# KPIs por escenario
for esc in ['Base','Adverso','Severamente Adverso']:
    subset = df[df['ESCENARIO'] == esc]
    st.subheader(f'Escenario: {esc}')
    cols = st.columns(4)
    for i, row in enumerate(subset.itertuples()):
        cols[i % 4].metric(row.SEGMENTO, f'{row.TASA_MORA_PROYECTADA}%', f'{row.PERDIDA_ESPERADA:,.0f}€')
    st.markdown('---')

st.subheader('Comparativa por Segmento')
seg = st.selectbox('Segmento', df['SEGMENTO'].unique())
df_seg = df[df['SEGMENTO'] == seg][['ESCENARIO','TASA_MORA_PROYECTADA','PERDIDA_ESPERADA']]
st.bar_chart(df_seg.set_index('ESCENARIO')['TASA_MORA_PROYECTADA'])

---

## Paso 8: Limpieza del Entorno (Opcional)

⚠️ **Advertencia**: Eliminará todos los datos y modelos.

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS STRESS_TESTING_DB;